# Whisper-large-v3 Yorùbá — test notebook

Companion to `Whisper.ipynb` (the training notebook). This one is **inference-only**:
no Unsloth, no LoRA wiring, no training deps. It loads a merged checkpoint from HF Hub
and evaluates it. Runs comfortably on a free Colab **T4** (≈4 GB VRAM at fp16).

**What it does**
1. Load the merged 16-bit checkpoint from `MODEL_ID` (defaults to your fine-tune).
2. Single-clip sanity check on a streamed FLEURS `yo_ng` sample (with audio playback).
3. WER on N FLEURS test clips — same setup as `scripts/eval_wer.py`.
4. Optional A/B vs base `openai/whisper-large-v3` so you can quantify the gain.
5. Optional: transcribe a file you upload yourself.

**Runtime**: any GPU with ≥6 GB VRAM. Colab T4 is fine. CPU works but is slow.

### Install

In [ ]:
%%capture
!pip install -q "transformers>=4.45" "datasets>=3.0" "huggingface_hub>=0.24" \
    librosa soundfile evaluate jiwer torchcodec accelerate

### HF auth (optional)

Only needed if the model repo is **private**. The fine-tune is public by default, so you
can skip this. If you do set `HF_TOKEN` in Colab Secrets (🔑 sidebar), this cell picks it up.

In [ ]:
import os

HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)
    print("HF auth: ok")
else:
    print("HF auth: skipped (public model OK)")

### Config

- `MODEL_ID` — the merged 16-bit checkpoint you pushed at the end of training.
- `BASELINE_ID` — `openai/whisper-large-v3` is the canonical baseline.
- `N_EVAL` — number of FLEURS clips. 50 for a sanity check, 200 for a real reading.

In [ ]:
MODEL_ID    = "devalade/whisper-large-v3-yoruba-colab"   # your fine-tune
BASELINE_ID = "openai/whisper-large-v3"                  # baseline for A/B
LANGUAGE    = "yoruba"
TASK        = "transcribe"
N_EVAL      = 50

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.float16 if DEVICE == "cuda" else torch.float32
print(f"device={DEVICE}  dtype={DTYPE}")

### Load the fine-tuned model

In [ ]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration

processor = WhisperProcessor.from_pretrained(MODEL_ID)
model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_ID, torch_dtype=DTYPE,
).to(DEVICE).eval()

# Force Yorùbá decoding — overrides whatever the checkpoint defaults to.
model.generation_config.language = "<|yo|>"
model.generation_config.task = TASK
model.generation_config.forced_decoder_ids = None

print(f"loaded {MODEL_ID}")

### Sanity check on one FLEURS clip

Streams a single Yorùbá clip from FLEURS, plays it back, and prints reference vs hypothesis.

In [ ]:
import soundfile as sf
from datasets import load_dataset, Audio
from IPython.display import Audio as AudioDisplay, display

fleurs = load_dataset("google/fleurs", "yo_ng", split="test", streaming=True)
sample = next(iter(fleurs.cast_column("audio", Audio(sampling_rate=16000))))
audio_path = "fleurs_yo_sample.wav"
sf.write(audio_path, sample["audio"]["array"], 16000)

ref = sample.get("transcription") or sample.get("raw_transcription") or ""

feats = processor.feature_extractor(
    sample["audio"]["array"],
    sampling_rate=16000,
    return_tensors="pt",
).input_features.to(DEVICE, dtype=DTYPE)

with torch.inference_mode():
    pred_ids = model.generate(feats, language="<|yo|>", task=TASK, max_new_tokens=256, num_beams=1)
hyp = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)[0]

print(f"Reference : {ref}")
print(f"Hypothesis: {hyp}")
display(AudioDisplay(audio_path, rate=16000))

### WER on FLEURS yo_ng (N clips)

Same setup as `scripts/eval_wer.py` — the WER number lands directly in the project's
experiments table. Run with `N_EVAL=200` for a number you can quote.

In [ ]:
import itertools
import evaluate
from tqdm.auto import tqdm
from datasets import load_dataset, Audio

wer_metric = evaluate.load("wer")
fleurs = (
    load_dataset("google/fleurs", "yo_ng", split="test", streaming=True)
    .cast_column("audio", Audio(sampling_rate=16000))
)

def transcribe(model, audio_array):
    feats = processor.feature_extractor(
        audio_array, sampling_rate=16000, return_tensors="pt",
    ).input_features.to(DEVICE, dtype=DTYPE)
    with torch.inference_mode():
        ids = model.generate(feats, language="<|yo|>", task=TASK, max_new_tokens=256, num_beams=1)
    return processor.tokenizer.batch_decode(ids, skip_special_tokens=True)[0]

refs, hyps = [], []
for s in tqdm(itertools.islice(fleurs, N_EVAL * 2), total=N_EVAL, desc=f"WER N={N_EVAL}"):
    if len(refs) >= N_EVAL:
        break
    r = s.get("transcription") or s.get("raw_transcription") or ""
    if not r.strip():
        continue
    refs.append(r)
    hyps.append(transcribe(model, s["audio"]["array"]))

wer = 100 * wer_metric.compute(predictions=hyps, references=refs)
print(f"\n{MODEL_ID}")
print(f"FLEURS yo_ng — N={len(refs)}  WER = {wer:.2f}%")

print("\n--- samples ---")
for i in range(min(5, len(refs))):
    print(f"REF[{i}]: {refs[i]}")
    print(f"HYP[{i}]: {hyps[i]}\n")

### Normalized WER — what the model is actually doing

Raw WER on Yorùbá is brutal because of three things that have **nothing to do with model quality**:

1. **Punctuation**: Whisper inserts `, . ?` and capitalizes. FLEURS references are lowercase and unpunctuated. Every comma attached to a word is a WER error.
2. **Diacritic conventions differ** between training data and FLEURS — and even *within* FLEURS itself, references are inconsistent (some samples have full tone marks, others have none).
3. **Word boundaries**: `torípé` ↔ `to rii pe` is three "errors" for the same content.

This is exactly why the Whisper paper applies `BasicTextNormalizer` to every non-English language before computing WER. We compute three numbers:

- **Raw WER** — what you saw above. Includes all the noise.
- **Normalized WER** — lowercase, strip punctuation, collapse whitespace. **This is the comparable number** — closest to what other Yorùbá ASR papers report.
- **Permissive WER** — also strips combining marks (`ọ→o`, `ẹ→e`, `ṣ→s`, tone marks). Tells you "did the model get the *consonants and vowels* right" independent of diacritics. Upper bound on content correctness.

In [ ]:
# Normalized + permissive WER on the refs/hyps from the previous cell.
import re
import unicodedata

_PUNCT_RE = re.compile(r"[^\w\s]", flags=re.UNICODE)
_WS_RE    = re.compile(r"\s+")

def normalize(s: str) -> str:
    """Lowercase, strip punctuation, collapse whitespace. Keeps diacritics."""
    s = s.lower()
    s = _PUNCT_RE.sub(" ", s)
    s = _WS_RE.sub(" ", s).strip()
    return s

def strip_diacritics(s: str) -> str:
    """NFD-decompose and drop combining marks. ọ→o, ẹ→e, ṣ→s, tone marks gone."""
    s = unicodedata.normalize("NFD", s)
    s = "".join(ch for ch in s if unicodedata.category(ch) != "Mn")
    return unicodedata.normalize("NFC", s)

assert refs and hyps, "Run the WER cell above first — refs/hyps must be populated."

refs_norm = [normalize(r) for r in refs]
hyps_norm = [normalize(h) for h in hyps]

refs_perm = [strip_diacritics(r) for r in refs_norm]
hyps_perm = [strip_diacritics(h) for h in hyps_norm]

wer_raw   = 100 * wer_metric.compute(predictions=hyps,      references=refs)
wer_norm  = 100 * wer_metric.compute(predictions=hyps_norm, references=refs_norm)
wer_perm  = 100 * wer_metric.compute(predictions=hyps_perm, references=refs_perm)

print(f"{MODEL_ID}  (FLEURS yo_ng, N={len(refs)})")
print(f"  raw WER         = {wer_raw:6.2f}%   ← what you saw before; noisy")
print(f"  normalized WER  = {wer_norm:6.2f}%   ← compare to published numbers")
print(f"  permissive WER  = {wer_perm:6.2f}%   ← no diacritics; content only")

print("\n--- normalized samples (what WER actually sees) ---")
for i in range(min(5, len(refs_norm))):
    print(f"REF[{i}]: {refs_norm[i]}")
    print(f"HYP[{i}]: {hyps_norm[i]}\n")

### Optional: A/B vs base `whisper-large-v3`

Loads the baseline next to your fine-tune and reports both WERs on the same N clips.
**Note**: this doubles GPU memory — on a T4 you may need to delete the fine-tuned model
first, run the baseline, and compare numbers offline. The cell below releases VRAM
between models so it fits in 15 GB.

In [ ]:
RUN_AB = False  # flip True to run the A/B

if RUN_AB:
    import gc
    refs_ab, hyps_ft = list(refs), list(hyps)  # already have fine-tune results
    ft_wer = wer

    # Free fine-tune from VRAM before loading baseline
    del model
    gc.collect(); torch.cuda.empty_cache()

    baseline = WhisperForConditionalGeneration.from_pretrained(
        BASELINE_ID, torch_dtype=DTYPE,
    ).to(DEVICE).eval()
    baseline.generation_config.language = "<|yo|>"
    baseline.generation_config.task = TASK
    baseline.generation_config.forced_decoder_ids = None

    fleurs2 = (
        load_dataset("google/fleurs", "yo_ng", split="test", streaming=True)
        .cast_column("audio", Audio(sampling_rate=16000))
    )
    hyps_base = []
    for s, r in zip(itertools.islice(fleurs2, N_EVAL * 2), refs_ab):
        if len(hyps_base) >= len(refs_ab):
            break
        hyps_base.append(transcribe(baseline, s["audio"]["array"]))

    base_wer = 100 * wer_metric.compute(predictions=hyps_base, references=refs_ab)
    print(f"\nN={len(refs_ab)}")
    print(f"  fine-tune ({MODEL_ID}): WER = {ft_wer:.2f}%")
    print(f"  baseline  ({BASELINE_ID}): WER = {base_wer:.2f}%")
    print(f"  Δ = {base_wer - ft_wer:+.2f} pp (positive = fine-tune wins)")
else:
    print("RUN_AB=False — flip the flag to compare against the baseline.")

### Optional: transcribe your own audio

Upload a `.wav` / `.mp3` / `.flac` file (16 kHz mono works best). Useful for ad-hoc tests
of how the fine-tune handles your specific accent, recording setup, or vocabulary.

In [ ]:
RUN_UPLOAD = False  # flip True to upload

if RUN_UPLOAD:
    import librosa
    from google.colab import files
    uploaded = files.upload()
    for fname in uploaded:
        audio, sr = librosa.load(fname, sr=16000, mono=True)
        hyp = transcribe(model, audio)
        print(f"\n{fname}")
        print(f"  → {hyp}")
else:
    print("RUN_UPLOAD=False — flip the flag and re-run to upload a file.")